# ChenLSTMISP-S: PerGAZE to OSIE subject adaptation on Colab (one GPU)

Select Runtime > Change runtime type > GPU. Load the trained PerGAZE ChenLSTMISP
checkpoint and OSIE images/test JSON from Drive. No DINOv2 features, text
embeddings, boxes or external attention maps are required. Inputs remain on
Drive; all checkpoints, reports and predictions persist there after each epoch.

Default K=5, seed=10 selects five test scanpaths per OSIE observer: 75 support
records and 975 remaining evaluation records for the supplied 1,050-record test.
Selected (image, observer) records are excluded from evaluation. Different
observers viewing the same image can occur on both sides. The split is identical
to Transfer/ChenLSTMISP-FT when K, seed and the source JSON match.

Only new OSIE subject embedding rows are optimized. The backbone, prediction
heads, original PerGAZE subject rows and all BatchNorm buffers remain unchanged.
The model stays in eval mode during adaptation, with autograd enabled for the
new embeddings. Adam uses LR=1e-3, zero weight decay and fresh optimizer state.
Default duration is 3 supervised epochs with train/eval batch=1. No RL.

Guidance is zero. TASK_HEAD=-1 reuses the VQA branch; 0..17 selects a COCO object
branch. This is branch selection, not a textual free-viewing instruction. Unlike
full FT, the chosen head remains frozen; only observer embeddings adapt.

Best epoch is selected by harmonic mean of ScanMatch with/without duration on
the remaining test. This is model-selected performance, not an untouched test
estimate. Use a separate ISP-S RUN; full-FT runs cannot resume as ISP-S.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

REPO_URL = "https://github.com/ntnghia06/IndividualScanpath.git"
CODE = Path("/content/IndividualScanpath")
# Optional: upload the local Transfer/ChenLSTMISP-S folder here when its code
# has not yet been pushed to GitHub. Only its src/ files are copied into Colab.
TRANSFER_SOURCE = Path("/content/drive/MyDrive/PerGAZE/transfer_code/ChenLSTMISP-S")

if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE)], check=True)
TRANSFER = CODE / "PerGAZE/Transfer/ChenLSTMISP-S"
if TRANSFER_SOURCE is not None and TRANSFER_SOURCE.is_dir():
    assert (TRANSFER_SOURCE / "src/train.py").is_file(), f"Invalid transfer source: {TRANSFER_SOURCE}"
    if (TRANSFER_SOURCE / "src").resolve() != (TRANSFER / "src").resolve():
        shutil.copytree(TRANSFER_SOURCE / "src", TRANSFER / "src", dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    print("Using transfer code from Drive:", TRANSFER_SOURCE)
SRC = TRANSFER / "src"
for name in ("train.py", "test.py", "fewshot.py", "osie_data.py", "options.py", "bootstrap.py"):
    assert (SRC / name).is_file(), (
        f"Missing {SRC / name}. Push the new transfer code to GitHub, or upload "
        "Transfer/ChenLSTMISP-S to TRANSFER_SOURCE on Drive and rerun this cell."
    )
assert (CODE / "PerGAZE/ChenLSTMISP/src/models/baseline_attention.py").is_file(), "Full PerGAZE base source is required"
print("Transfer source:", SRC)


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze scikit-image


In [ ]:
import torch

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Drive paths and hyperparameters

Copy `D:/PerScan/Experiments/osie` to Drive, or edit DATA to its existing location.
Only `osie_fixations_test.json` and `images/` are read. The default layout is:

```text
MyDrive/PerGAZE/
  osie/
    osie_fixations_test.json
    images/1009.jpg ...
  runs/pergaze_chenlstm_colab_1gpu/checkpoints/best.pth
  transfer_code/ChenLSTMISP-S/src/  # optional source override from the setup cell
```

BASE_RUN points to the actual trained PerGAZE run, not an OSIE or functional-test
checkpoint. The checkpoint needs separate TP/TA observer identities, 18 COCO
object branches and the VQA branch. Architecture/max_length come from it.

K is the number of support scanpaths **per observer**. Sampling uses the OSIE
test JSON exclusively; OSIE train and validation are not loaded. INIT="mean"
initializes each new subject from the mean of all PerGAZE observer embeddings.

RESUME=False starts a new run. Set RESUME=True after an interruption and keep
the same base checkpoint, source test JSON, K, seed, head, initialization, LR,
batches, evaluation settings and batch limits. Increase EPOCHS to extend a run.

LEARNING_RATE defaults to 1e-3 for subject-only adaptation. WEIGHT_DECAY must
remain 0 so source observer rows stay unchanged. Backbone/head weights are frozen.


In [ ]:
import json
import math
from collections import Counter

K = 5
SEED = 10
EVAL_SEED = 10
EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GRAD_CLIP = 12.5
TRAIN_BATCH = 1
TEST_BATCH = 1
WORKERS = 2
TASK_HEAD = -1  # -1=VQA branch; 0..17=COCO object branch. Not a free-viewing label.
INIT = "mean"  # "mean" or "random"
EVAL_REPEAT_NUM = 1
GREEDY = False
MAX_BATCHES = 0       # 0=all support; 1=one-batch functional check
EVAL_MAX_BATCHES = 0  # 0=all remaining test; 1=one-batch functional check
RESUME = False

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA = DRIVE_ROOT / "osie"
TEST_FILE = DATA / "osie_fixations_test.json"
IMAGE_DIR = DATA / "images"
BASE_RUN = DRIVE_ROOT / "runs/pergaze_chenlstm_colab_1gpu"
CHECKPOINT = BASE_RUN / "checkpoints/best.pth"
RUN = DRIVE_ROOT / "runs" / f"osie_chenlstm_s_k{K}_seed{SEED}_head{TASK_HEAD}_colab_1gpu"
LOCAL_ROOT = Path("/content/osie_chenlstm_s_work")  # Subprocess logs / optional ZIP only.

assert TEST_FILE.is_file(), f"Missing OSIE test JSON: {TEST_FILE}"
assert IMAGE_DIR.is_dir(), f"Missing OSIE images: {IMAGE_DIR}"
assert CHECKPOINT.is_file(), f"Missing trained PerGAZE ChenLSTM checkpoint: {CHECKPOINT}"
assert min(K, EPOCHS, TRAIN_BATCH, TEST_BATCH, EVAL_REPEAT_NUM) >= 1
assert math.isfinite(LEARNING_RATE) and LEARNING_RATE > 0
assert math.isfinite(WEIGHT_DECAY) and WEIGHT_DECAY >= 0
assert math.isfinite(GRAD_CLIP) and min(WORKERS, MAX_BATCHES, EVAL_MAX_BATCHES) >= 0
assert INIT in ("mean", "random") and TASK_HEAD in range(-1, 18)
assert 0 <= SEED < 2**32 and 0 <= EVAL_SEED < 2**32
rows = json.loads(TEST_FILE.read_text(encoding="utf-8-sig"))
assert rows and all(row.get("split") == "test" for row in rows), "Use the OSIE test JSON only"
counts = Counter(row["subject"] for row in rows)
assert all(count > K for count in counts.values()), "K must leave evaluation records for every subject"
missing = sorted({row["name"] for row in rows if not (IMAGE_DIR / row["name"]).is_file()})
assert not missing, f"Missing {len(missing)} OSIE images; first: {missing[:5]}"
RUN.mkdir(parents=True, exist_ok=True)
last_checkpoint = RUN / "checkpoints/checkpoint.pth"
if RESUME:
    assert last_checkpoint.is_file(), f"No transfer checkpoint at {last_checkpoint}"
else:
    assert not last_checkpoint.exists(), "Existing run: set RESUME=True or choose a new RUN"
print("OSIE observers:", len(counts), "source test samples:", len(rows))
print("Selected support:", K * len(counts), "remaining evaluation:", len(rows) - K * len(counts))
print("Task head:", TASK_HEAD, "guidance: zero", "embedding-only adaptation: True")
print("Drive input:", DATA)
print("PerGAZE base checkpoint:", CHECKPOINT)
print("Drive output:", RUN)


In [ ]:
import shutil

COMMON = [
    "--checkpoint", str(CHECKPOINT),
    "--data_root", str(DATA), "--test_file", str(TEST_FILE), "--img_dir", str(IMAGE_DIR),
    "--k", str(K), "--seed", str(SEED), "--eval_seed", str(EVAL_SEED),
    "--epoch", str(EPOCHS), "--lr", str(LEARNING_RATE),
    "--weight_decay", str(WEIGHT_DECAY), "--clip", str(GRAD_CLIP),
    "--task_head", str(TASK_HEAD), "--init", INIT,
    "--eval_repeat_num", str(EVAL_REPEAT_NUM),
    "--max_batches", str(MAX_BATCHES), "--eval_max_batches", str(EVAL_MAX_BATCHES),
    "--device", "cuda:0", "--gpu_ids", "0",
]
if GREEDY:
    COMMON.append("--greedy")

def run_script(script, *arguments):
    import codecs
    import time

    command = [sys.executable, "-u", str(SRC / script), *COMMON, *arguments]
    local_logs = LOCAL_ROOT / "logs"
    local_logs.mkdir(parents=True, exist_ok=True)
    log_path = local_logs / f"{Path(script).stem}_{time.time_ns()}.log"
    drive_log = RUN / "logs" / log_path.name
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = bytearray()
    process = None
    return_code = None
    print("Full subprocess log:", log_path, flush=True)
    try:
        with log_path.open("wb") as log:
            log.write(("Command: " + repr(command) + "\n").encode("utf-8"))
            process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, cwd=str(CODE))
            try:
                while True:
                    chunk = process.stdout.read1(8192)
                    if not chunk:
                        break
                    log.write(chunk)
                    log.flush()
                    tail.extend(chunk)
                    if len(tail) > 65536:
                        del tail[:-65536]
                    sys.stdout.write(decoder.decode(chunk))
                    sys.stdout.flush()
                sys.stdout.write(decoder.decode(b"", final=True))
                return_code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                process.stdout.close()
    finally:
        if log_path.is_file():
            try:
                drive_log.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(log_path, drive_log)
                print("Log copied to Drive:", drive_log, flush=True)
            except OSError as error:
                print("Could not copy log to Drive; local log is retained:", error, flush=True)
    if return_code:
        lines = tail.decode("utf-8", errors="replace").replace("\r", "\n").splitlines()
        last_lines = "\n".join(lines[-40:])
        hint = ""
        if "out of memory" in last_lines.lower():
            hint = "\nMemory error: for a NEW run try TRAIN_BATCH=1, TEST_BATCH=1. Resume requires the saved train batch."
        elif "checkpoint exists" in last_lines.lower():
            hint = "\nSet RESUME=True to continue that run, or choose a new RUN directory."
        raise RuntimeError(f"{script} exited with code {return_code}. Full log: {log_path}"
                           f"\nLast subprocess output:\n{last_lines}{hint}")


## Prepare support and run embedding-only adaptation

Preparation validates the OSIE JSON and writes support.json, test_remaining.json
and selection.json directly to RUN without loading weights. It is skipped on
resume. Training reconstructs and verifies the same deterministic selection.

Baseline evaluation runs before adaptation. Each epoch optimizes only new OSIE
observer rows, then evaluates the remaining test and saves last/best checkpoint,
report and predictions. All other weights, dropout and BatchNorm buffers remain
frozen. Original subject rows are checked after every epoch. Set batch limits
only for functional checks; the report flags incomplete runs.

The runner streams stdout/stderr and copies logs with original tracebacks to
RUN/logs. A partially completed epoch restarts from the last completed epoch.
No image feature cache or dataset copy is created.


In [ ]:
if not RESUME:
    run_script("train.py", "--prepare_only", "--log_root", str(RUN), "--workers", "0")
    split_info = json.loads((RUN / "selection.json").read_text(encoding="utf-8"))
    print("Saved support:", split_info["support_samples"])
    print("Saved remaining test:", split_info["remaining_test_samples"])
else:
    print("Resume: preserve the existing split; training will verify selection/provenance.")

arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


In [ ]:
report = json.loads((RUN / "report.json").read_text(encoding="utf-8"))
print("Selected support samples:", report["support_samples"])
print("Remaining test samples:", report["test_samples"])
print("OSIE subjects:", report["adaptive_subjects"])
print("Task head:", report["task_head"], "guidance:", report["guidance"])
print("Best epoch:", report["best_epoch"], "score:", report["best_score"])
print(json.dumps(report["best_metrics"]["overall"], indent=2))
print(json.dumps(report["best_metrics"].get("retrieval", {}).get("by_condition", {}), indent=2))
print("Full support trained:", report["full_selected_support_training"])
print("Full remainder evaluated:", report["full_unseen_evaluation"])
print("Full report:", RUN / "report.json")


## Optional reevaluation

Uncomment the next cell to reevaluate the adapted best.pth on the same remaining
OSIE test. The final --checkpoint argument overrides the base checkpoint in
COMMON. test.py restores support seed/K/head/evaluation settings from the adapted
checkpoint and verifies the original test JSON hash. Selected support records
are excluded again. Results append to report.json and predictions_reevaluation.json.


In [ ]:
# run_script("test.py", "--checkpoint", str(RUN / "checkpoints/best.pth"),
#            "--log_root", str(RUN), "--workers", str(WORKERS),
#            "--test_batch", str(TEST_BATCH))


## Resume and outputs

After a disconnect, mount Drive, rerun setup/configuration with RESUME=True and
the same RUN and original base checkpoint. Resume restores model, optimizer and
RNG state and rejects changed source data, selected support or training settings.
Do not reuse an OSIE full-FT or PerGAZE adaptation run directory as this OSIE ISP-S RUN.

Drive outputs:
- support.json, test_remaining.json, selection.json (source hash and selected IDs).
- checkpoints/checkpoint.pth (last) and checkpoints/best.pth.
- report.json (baseline, history, metrics, predictions, retrieval and protocol).
- predictions_best.json and predictions_last.json: X/Y in original image pixels,
  T in milliseconds; scanpath retains model-space coordinates/seconds for metrics.
- logs/ containing subprocess stdout/stderr.

The next cell optionally downloads a ZIP. Checkpoints already persist on Drive;
creating a ZIP can take time and additional Colab disk space.


In [ ]:
print("Outputs saved on Drive:", RUN)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# archive = shutil.make_archive(str(LOCAL_ROOT / f"osie_chenlstm_s_k{K}_seed{SEED}_head{TASK_HEAD}"),
#                              "zip", root_dir=RUN.parent, base_dir=RUN.name)
# from google.colab import files
# files.download(archive)
